In [9]:
!pip install modal

In [10]:
import os
from google.colab import userdata

try:
    os.environ["MODAL_TOKEN_ID"] = userdata.get('token-id')
    os.environ["MODAL_TOKEN_SECRET"] = userdata.get('token-secret')
    print("✅ Authentifizierung für Modal geladen!")
except Exception as e:
    print(f"❌ Fehler: {e}. Hast du das Schlüssel-Icon links konfiguriert?")

tid = os.environ.get("MODAL_TOKEN_ID")
tsec = os.environ.get("MODAL_TOKEN_SECRET")
print(f"Token ID startet mit: {tid[:4]}... (Länge: {len(tid)})")
print(f"Token Secret startet mit: {tsec[:4]}... (Länge: {len(tsec)})")



✅ Authentifizierung für Modal geladen!
Token ID startet mit: ak-b... (Länge: 25)
Token Secret startet mit: as-p... (Länge: 25)


In [11]:
import os
from google.colab import userdata

# 1. Token aus den Colab-Secrets laden
hf_token_value = userdata.get('HF_TOKEN')

# 2. Das Modal-Secret erstellen, ohne den Token im Code zu zeigen
# Wir nutzen die f-String Syntax für den System-Befehl
if hf_token_value:
    !modal secret create vbot_modal_huggingface HF_TOKEN='{hf_token_value}' --force
    print("✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!")
else:
    print("❌ Fehler: HF_TOKEN wurde nicht in den Colab-Secrets gefunden.")

Created a new secret 'vbot_modal_huggingface' with the key 'HF_TOKEN'

Use it in your Modal app:

                                                                                
@app.function(secrets=[modal.Secret.from_name("vbot_modal_huggingface")])       
def some_function():                                                            
    os.getenv("HF_TOKEN")                                                       
                                                                                
✅ Modal Secret 'vbot_modal_huggingface' wurde erfolgreich erstellt!


In [12]:

!modal profile list

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━┓
┃   ┃ Profile ┃ Workspace ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━┩
└───┴─────────┴───────────┘
Using matthias-nollek workspace based on environment variables


In [24]:
%%writefile app.py
import modal
import os
import uuid
import asyncio
import json
from fastapi import FastAPI, WebSocket
from starlette.responses import HTMLResponse

# 1. Konfiguration
MAX_TOKENS = 8192
MAX_NEW_TOKENS = 2048
MODEL_ID = "mistralai/Mistral-Nemo-Instruct-FP8-2407"
SYSTEM_PROMPT = "You are a helpful assistant..."

# Define the model path on the volume
MODEL_VOLUME_PATH = "/data/mistral-nemo"

# Create (or reference) a Volume named "model-storage"
model_volume = modal.Volume.from_name("model-storage", create_if_missing=True)

# 2. Image Definition
vllm_image = (
    modal.Image.debian_slim(python_version="3.10")
    .pip_install(
        "vllm>=0.6.0",
        "transformers>=4.44.0",
        "tokenizers>=0.19.0",
        "fastapi",
        "starlette",
        "hf_transfer",
        "huggingface_hub"
    )
    .env({
        "HF_HUB_ENABLE_HF_TRANSFER": "1",
        "VLLM_LOGGING_LEVEL": "ERROR", # Changed from ERROR to INFO for more verbosity
        "VLLM_USE_V1": "0"
    })
)

app = modal.App("twilio-voice-nemo")

# 3. Der Haupt-Service
@app.cls(
    image=vllm_image,
    gpu="A10G",
    secrets=[modal.Secret.from_name("vbot_modal_huggingface")],
    volumes={"/data": model_volume}, # Mount the volume
    scaledown_window=10
)
class TwilioChatBot:
    @modal.enter()
    def load_engine(self):
        from vllm import AsyncEngineArgs, AsyncLLMEngine
        from transformers import AutoTokenizer
        from huggingface_hub import snapshot_download

        # Check if model is on the volume, download if not
        if not os.path.exists(MODEL_VOLUME_PATH):
            print("📥 Model will be downloaded to the volume for the first time...")
            snapshot_download(
                MODEL_ID, # Use the original MODEL_ID for download
                local_dir=MODEL_VOLUME_PATH,
                ignore_patterns=["*.pt", "*.bin"] # Save space if SafeTensors are present
            )
            # IMPORTANT: Changes to the volume must be committed
            model_volume.commit()

        # Now vLLM loads locally from the volume instead of from the network
        engine_args = AsyncEngineArgs(
            model=MODEL_VOLUME_PATH, # Path on the volume
            gpu_memory_utilization=0.90,
            max_model_len=MAX_TOKENS,
            trust_remote_code=True,
            enforce_eager=True
        )
        self.engine = AsyncLLMEngine.from_engine_args(engine_args)
        # WICHTIG: Tokenizer der Klasse zuweisen
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_VOLUME_PATH) # Load tokenizer from volume

    @modal.asgi_app()
    def fastapi_app(self):
        web_app = FastAPI()

        @web_app.post("/start_call")
        async def start_call(request: Request):
            # 1. Dynamische URL-Auflösung über das Request-Objekt
            # Dies extrahiert automatisch den Hostname (z.B. user--app-name.modal.run)
            host = request.url.netloc

            # 2. Das XML-Template mit der korrekten wss:// URL
            # Wir nutzen f-Strings und achten darauf, dass die WebSocket-URL passt
            tmpl = f"""<?xml version="1.0" encoding="UTF-8"?>
        <Response>
          <Connect>
            <ConversationRelay
                url="wss://{host}/ws"
                welcomeGreeting="Hi! I'm Jane. Just chat with me!!">
            </ConversationRelay>
          </Connect>
        </Response>"""

        return HTMLResponse(content=tmpl, media_type="application/xml")

        @web_app.websocket("/ws")
        async def websocket_endpoint(websocket: WebSocket):
            await websocket.accept()
            queue = asyncio.Queue()
            # Initialer Verlauf mit System Prompt
            history = [{"role": "system", "content": SYSTEM_PROMPT}]

            async def llm_request(message):
                from vllm import SamplingParams
                sampling_param = SamplingParams(max_tokens=MAX_NEW_TOKENS)

                prompt = self.tokenizer.apply_chat_template(
                    history + [dict(role="user", content=message)],
                    tokenize=False, add_generation_prompt=True,
                )
                request_id = uuid.uuid4().hex
                replies = []

                try:
                    stream = await self.engine.add_request(request_id, prompt, sampling_param)
                    cursor = 0
                    async for request_output in stream:
                        text = request_output.outputs[0].text
                        out = text[cursor:]
                        replies.append(out)
                        await websocket.send_json({"type": "text", "token": out, "last": False})
                        cursor = len(text)
                except asyncio.CancelledError:
                    await self.engine.abort(request_id)
                    raise
                finally:
                    reply = "".join(replies)
                    history.append(dict(role="user", content=message))
                    history.append(dict(role="assistant", content=reply))
                    await websocket.send_json({"type": "text", "token": "", "last": True})

            async def read_from_socket():
                async for data in websocket.iter_json():
                    await queue.put(data)

            async def process_logic():
                input_buffer = []
                llm_task = None
                while True:
                    data = await queue.get()
                    if data["type"] == "prompt":
                        input_buffer.append(data["voicePrompt"])
                        if data.get("last"):
                            message = " ".join(input_buffer)
                            input_buffer = []
                            if llm_task: llm_task.cancel()
                            llm_task = asyncio.create_task(llm_request(message))
                    elif data["type"] == "interrupt":
                        input_buffer = []
                        if llm_task: llm_task.cancel()

            await asyncio.gather(read_from_socket(), process_logic())

        return web_app

Overwriting app.py


In [25]:
!python3 -m py_compile app.py

In [26]:
!modal deploy app.py

⠸ Creating objects...
⠦ Creating objects...
⠏ Creating objects...
⠹ Creating objects...
⠴ Creating objects...
├── 🔨 Created mount /content/app.py
├── 🔨 Created function TwilioChatBot.*.
└── 🔨 Created web endpoint for TwilioChatBot.fastapi_app => 
    https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.r
    un
✓ Created objects.
├── 🔨 Created mount /content/app.py
├── 🔨 Created function TwilioChatBot.*.
└── 🔨 Created web endpoint for TwilioChatBot.fastapi_app => 
    https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.r
    un
✓ App deployed in 1.659s! 🎉

View Deployment: 
https://modal.com/apps/matthias-nollek/main/deployed/twilio-voice-nemo
^C


In [23]:
!curl -X POST https://matthias-nollek--twilio-voice-nemo-twiliochatbot-fastapi-app.modal.run/start_call \
     -H "Content-Type: application/json" \
     -d '{"prompt": "Wer bist du?"}'

Internal Server Error